# Chapter 29: a backtesting ledger

Run two assets plus cash on four dates with lag 1 and fee rate 0.002, then check drift, rebalancing and reconciliation. The shared constructed study compares drift-aware vectorization with the event ledger under zero fees and E=P, and tests ruin, splits, delisting and future perturbations. French full-sample 1/N analysis is skipped. RUN_DIR is allocated, but no result files or new plots are saved.

[Entry modes and preparation](../NOTEBOOK_GUIDE.md). This stage delivers language views; portable dependencies and execution are checked in the later isolated cold start.

In [ ]:
from pathlib import Path
import sys, os, uuid
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start inside the reader package tree")
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.environ["MPLBACKEND"] = "Agg"
RUN_DIR = ROOT / "runs" / ("ch29_backtest_engine_" + uuid.uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
os.chdir(ROOT)
# French branches remain disabled until data rights and execution authorization are resolved.
RUN_FRENCH = False
print("CONSTRUCTED teaching route; French market branches SKIPPED; outputs:", RUN_DIR)

import sys
from pathlib import Path

sys.path.insert(0, str(ROOT / "src"))
print("interpreter:", sys.executable)
print("project root:", ROOT)
import importlib.util


def load_code_module(filename: str):
    """Import a chapter CLI from code/ so the notebook calls exactly the functions the CLI runs."""
    spec = importlib.util.spec_from_file_location(Path(filename).stem, ROOT / "code" / filename)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

## 1. Targets to after-fee holdings

Execution and mark prices are separate tables. The first post-fee NAV is 1000/1.0016; shares and cash determine the second rebalance's pre/post-fee NAV. Inspect ledger, fills, positions, weights and reconciliation separately.

In [ ]:
import numpy as np
import pandas as pd
from quant_course.backtest import BacktestConfig, run_backtest, reconcile, solve_post_fee_nav

dates = pd.DatetimeIndex(["2026-01-30", "2026-02-02", "2026-02-27", "2026-03-02"])
mark = pd.DataFrame({"A": [10.0, 11.0, 12.1, 12.0], "B": [20.0, 19.0, 19.0, 21.0]}, index=dates)
execp = pd.DataFrame({"A": [10.0, 10.0, 12.0, 12.0], "B": [20.0, 20.0, 19.5, 20.0]}, index=dates)
tw = pd.DataFrame({"A": [0.5, 0.5], "B": [0.3, 0.3]}, index=[dates[0], dates[2]])
cfg = BacktestConfig(initial_cash=1000.0, fee_rate=0.002, lag=1)

res = run_backtest(mark, tw, cfg, exec_prices=execp)
rec = reconcile(res, mark, cfg, exec_prices=execp)

# Hand-solved closed forms:
# Rebalance 1:
v1 = solve_post_fee_nav(1000.0, np.zeros(2), np.array([0.5, 0.3]), 0.002)
assert np.isclose(v1, 1000.0 / 1.0016)
print(f"Rebalance 1 post-fee NAV V': {v1:.4f} (fees: {1000.0 - v1:.4f})")

# Rebalance 2:
qa1, qb1 = 0.5 * v1 / 10.0, 0.3 * v1 / 20.0
cash1 = 1000.0 - (qa1 * 10.0 + qb1 * 20.0) - 0.002 * (qa1 * 10.0 + qb1 * 20.0)
v2_pre = cash1 + qa1 * 12.0 + qb1 * 20.0
v2 = solve_post_fee_nav(v2_pre, np.array([qa1 * 12.0, qb1 * 20.0]), np.array([0.5, 0.3]), 0.002)
print(f"Rebalance 2 pre-trade NAV: {v2_pre:.4f}, post-fee NAV: {v2:.4f} (fees: {v2_pre - v2:.4f})")

print("\n--- Backtest Ledger ---")
display(res.ledger.round(4))

In [ ]:
print("--- Trade Fills Table ---")
display(res.fills.round(4))
print("\n--- Positions (Shares) ---")
display(res.positions.round(5))
print("\n--- Holding Weights (Fraction of NAV) ---")
display(res.weights.round(4))
print("\n--- Reconciliation Breakdown ---")
display(rec.round(6))
assert rec["residual"].abs().max() < 1e-12
assert rec["cash_identity"].abs().max() < 1e-12

## 2. Drift and turnover

Compare $w_i(1+r_i)/(1+r_p)$ with held weights, then compute buys-plus-sells turnover from pre-trade drifted weights to targets. Cash has its own return and weight.

In [ ]:
from quant_course.portfolio_construction import drift_weights, turnover

# Compare drift formula w_i * (1 + r_i) / (1 + r_p) with ledger actual weights
w_init = np.array([0.53140097, 0.27536232, 0.19323671])  # A, B, cash on 02-02
r_asset = np.array([12.1 / 11.0 - 1.0, 19.0 / 19.0 - 1.0, 0.0])  # return 02-02 to 02-27
w_drifted = drift_weights(w_init, r_asset)
print(f"Drifted weights on 02-27: A={w_drifted[0]:.4f}, B={w_drifted[1]:.4f}, Cash={w_drifted[2]:.4f}")
assert np.isclose(w_drifted[0], res.weights.loc["2026-02-27", "A"], atol=1e-4)

# Turnover between drifted pre-trade weights and target weights
pre_weights_0302 = np.array([qa1 * 12.0 / v2_pre, qb1 * 20.0 / v2_pre, cash1 / v2_pre])
target_w = np.array([0.5, 0.3, 0.2])
to = turnover(pre_weights_0302, target_w)
print(f"Rebalance turnover: {to:.4f}")

## 3. Vectorization and ledger boundary

With zero fees, zero rates and execution price equal to mark price, the drift-aware formula agrees with the ledger. A naive shifted-target formula ignores drift and produces tracking error; this does not validate arbitrary vectorized backtests.

In [ ]:
from quant_course.backtest import drifted_weight_returns

ch29 = load_code_module("ch29_backtest_engine.py")
cons = ch29.constructed_study()

print("Vectorized with drift vs Ledger (zero fee, E=P) max absolute difference:")
print(f"  {cons['vectorized_vs_ledger_max_diff']:.2e}")
assert cons["vectorized_vs_ledger_max_diff"] < 1e-12

print("\nNaive Shift-only Vectorized vs Drifted Vectorized max difference:")
print(f"  {cons['naive_vs_drifted_max_diff']:.6f}")
assert cons["naive_vs_drifted_max_diff"] > 1e-4
print("Naive shift ignores within-period drift, causing tracking errors!")

## 4. Boundary policies

Inspect NAV≤0 termination, split value neutrality, delisting cash and future-input invariance. These are specified policies, not a model of every corporate action or execution rule.

In [ ]:
pol = cons["policies"]
print("1. Ruin Termination (NAV <= 0):")
print(f"   Terminated at: {pol['ruin_terminated_at']}, Final NAV: {pol['ruin_final_nav']:.2f}")

print("\n2. Corporate Actions (Stock Split):")
print(f"   NAV with 2:1 split booked: ${pol['split_nav_with_event']:.2f} (Value-neutral)")
print(f"   NAV without split booked:  ${pol['split_nav_without_event']:.2f} (Artificial -50% loss!)")

print("\n3. Delisting Cash-Out:")
print(f"   Delist cash received: ${pol['delist_cash_received']:.2f}, Final NAV: ${pol['delist_final_nav']:.2f}")
print(f"   Max reconciliation residual: {pol['delist_max_residual']:.2e}")

print("\n4. Future Perturbation Invariance:")
print(f"   Past ledger strictly unchanged: {pol['future_perturbation_past_identical']}")

## 5. French historical 1/N: skipped

The conditional block preserves a historical cost-sensitivity comparison. No 1,202-month market result is reproduced by default.

In [ ]:
if RUN_FRENCH:
    real = ch29.real_study()
    print("Dataset:", real["dataset"])
    print(f"Sample: {real['first_month']} to {real['last_month']} ({real['months']} months)")
    print(f"Processed SHA-256: {real['manifest_sha256'][:16]}...{real['manifest_sha256'][-8:]}")
    print(f"\nZero-fee Ledger vs Vectorized with drift discrepancy on real data: {real['zero_fee_ledger_vs_vectorized_diff']:.2e}")
    assert real["zero_fee_ledger_vs_vectorized_diff"] < 1e-12
    
    print(f"Annual rebalancing naive shift vs drifted max difference: {real['annual_rebalance_naive_vs_drift_max_diff']:.4f}")
    assert real["annual_rebalance_naive_vs_drift_max_diff"] > 0.01
    
    print("\n--- Fee Rate Sensitivity Table (French 10 Industries 1/N) ---")
    fee_df = pd.DataFrame(real["fee_sensitivity"]).T[["fee_rate", "cagr", "annualized_vol", "annualized_sharpe", "total_fees", "avg_monthly_turnover", "annual_fee_drag"]]
    fee_df["cagr"] = (fee_df["cagr"] * 100).round(2).astype(str) + "%"
    fee_df["annualized_vol"] = (fee_df["annualized_vol"] * 100).round(2).astype(str) + "%"
    fee_df["annualized_sharpe"] = fee_df["annualized_sharpe"].round(3)
    fee_df["avg_monthly_turnover"] = (fee_df["avg_monthly_turnover"] * 100).round(2).astype(str) + "%"
    fee_df["annual_fee_drag (bps)"] = (fee_df["annual_fee_drag"] * 10000).round(1).astype(str)
    fee_df["total_fees"] = fee_df["total_fees"].round(2)
    display(fee_df)
else:
    print("SKIPPED French branch: permissions unknown; no bundled data; historical empirical results not reproduced")


## 6. Historical gallery skipped

The next cell prints a skip message rather than displaying historical plots.

In [ ]:
print("SKIPPED historical figure gallery; constructed calculations above are freshly computed; French historical images are not reproduced")


## 7. Questions

1. Set lag=2 in a disposable constructed copy and trace signal/execution misalignment through fills and reconciliation.
2. $c\sum|w|<1$ is a sufficient strict-monotonicity/uniqueness condition. Why does its failure not prove that the general post-fee equation has no financial positive root? Distinguish multiple, absent and interval roots from this API's strict-domain policy; see 29-E03.
3. Why is a total-return index unsuitable as an unadjusted trade price in an event ledger?